In [1]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
from scipy.stats import entropy
from scipy.optimize import curve_fit
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. 追加検証シミュレーション設定
# ==========================================
N_TRIALS = 1000

DIMENSIONS = [10, 50, 100, 200]
# 遷移領域を高解像度で捉えるため、対数空間で細かく分割 (例: 10^3.5 〜 10^6.5)
SAMPLE_SIZES = np.logspace(3.5, 6.5, num=40, dtype=int)
EPSILON_THRESHOLD = 0.001

output_dir = "hypothesis3_phase_transition"
os.makedirs(output_dir, exist_ok=True)

print("崩壊曲線の相転移・加速度分析シミュレーションを開始します...")

# ==========================================
# 2. 単一試行関数 (H3統合DGP)
# ==========================================
def simulate_h3_single_trial(D, N):
    K = 5
    cell_raw = np.random.uniform(0, 0.2, D)
    P_cell = cell_raw / np.sum(cell_raw)

    betas = 10 ** np.random.uniform(-1, 1.5, D)
    X_choice = np.random.uniform(0, 0.2, (D, K))

    P_joint = np.zeros((D, K))
    for d in range(D):
        exp_val = np.exp(betas[d] * X_choice[d])
        P_choice_given_d = exp_val / np.sum(exp_val)
        P_joint[d, :] = P_cell[d] * P_choice_given_d

    P_flat = P_joint.flatten()

    C_flat = np.random.multinomial(N, P_flat)
    Q_flat = C_flat / N
    Q_flat = np.where(Q_flat == 0, 1e-10, Q_flat)

    kl_div = entropy(P_flat, Q_flat)
    is_collapsed = 1 if kl_div > EPSILON_THRESHOLD else 0

    return is_collapsed

# ==========================================
# 3. 高解像度グリッド探索
# ==========================================
results_summary = []

for D in DIMENSIONS:
    print(f"\n--- 次元数 D = {D} の相転移スキャン ---")
    for N_samp in SAMPLE_SIZES:
        trial_results = Parallel(n_jobs=-1)(
            delayed(simulate_h3_single_trial)(D, N_samp) for _ in range(N_TRIALS)
        )
        collapse_rate = np.sum(trial_results) / N_TRIALS

        results_summary.append({
            "Dimension_D": D,
            "Sample_Size_N": N_samp,
            "Log10_N": np.log10(N_samp),
            "Collapse_Rate": collapse_rate
        })

df_res = pd.DataFrame(results_summary)
df_res.to_csv(f"{output_dir}/phase_transition_raw_data.csv", index=False)

# ==========================================
# 4. シグモイド関数フィッティングと加速度(κ)の計算
# ==========================================
# 崩壊曲線モデル: y = 1 / (1 + exp(kappa * (x - x50)))
def sigmoid(x, kappa, x50):
    # オーバーフロー回避
    z = np.clip(kappa * (x - x50), -100, 100)
    return 1 / (1 + np.exp(z))

fit_results = []
x_fit = np.linspace(3.5, 7.0, 200)
dict_fits = {}

for D in DIMENSIONS:
    subset = df_res[df_res["Dimension_D"] == D]
    x_data = subset["Log10_N"].values
    y_data = subset["Collapse_Rate"].values

    try:
        # 初期値推定 (kappa=5.0, x50は中点あたり)
        p0 = [5.0, np.mean(x_data)]
        popt, pcov = curve_fit(sigmoid, x_data, y_data, p0=p0, bounds=([0.1, 3.0], [50.0, 7.0]))
        kappa, x50 = popt

        # 崩壊が始まるポイント(95%崩壊)と、終わるポイント(5%崩壊)のギャップ幅(Δx)
        # 1/(1+exp(k(x-x50))) = 0.95 -> exp(k(x-x50)) = 1/19 -> x_95 = x50 - ln(19)/k
        # 1/(1+exp(k(x-x50))) = 0.05 -> exp(k(x-x50)) = 19 -> x_05 = x50 + ln(19)/k
        delta_x = (2 * np.log(19)) / kappa

        fit_results.append({
            "Dimension_D": D,
            "Acceleration_Kappa": kappa,
            "Half_Life_Log10_N50": x50,
            "Transition_Width_DeltaLogN": delta_x
        })
        dict_fits[D] = sigmoid(x_fit, kappa, x50)

    except Exception as e:
        print(f"D={D} のフィッティングに失敗しました: {e}")

df_fit = pd.DataFrame(fit_results)
df_fit.to_csv(f"{output_dir}/phase_transition_metrics.csv", index=False)
display(df_fit)

# ==========================================
# 5. 可視化
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')

# グラフ1: 崩壊曲線のフィッティングと相転移の鋭さ
plt.figure(figsize=(10, 6))
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']

for idx, D in enumerate(DIMENSIONS):
    subset = df_res[df_res["Dimension_D"] == D]
    plt.scatter(subset["Log10_N"], subset["Collapse_Rate"],
                color=colors[idx], alpha=0.5, s=20)
    if D in dict_fits:
        plt.plot(x_fit, dict_fits[D], color=colors[idx], linewidth=2,
                 label=f'D={D} ($\kappa$={df_fit[df_fit["Dimension_D"]==D]["Acceleration_Kappa"].values[0]:.2f})')

plt.axhline(y=0.05, color='gray', linestyle='--', label='Robust Threshold (5%)')
plt.title("Phase Transition of Statistical Collapse (Sigmoid Fit)")
plt.xlabel("Sample Size $\log_{10}(N)$")
plt.ylabel("Collapse Rate")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig3_phase_transition_curves.png", dpi=300)
plt.close()

# グラフ2: 次元増大に伴う崩壊加速度(κ)の上昇
plt.figure(figsize=(8, 5))
plt.plot(df_fit["Dimension_D"], df_fit["Acceleration_Kappa"], marker='D', color='purple', linewidth=2)
plt.title("Increase in Collapse Acceleration ($\kappa$) by Dimension")
plt.xlabel("Categorical Dimension D")
plt.ylabel("Acceleration Parameter ($\kappa$)")
plt.tight_layout()
plt.savefig(f"{output_dir}/fig4_acceleration_kappa.png", dpi=300)
plt.close()

# ==========================================
# 6. ZIP化とダウンロード
# ==========================================
zip_filename = "hypothesis3_phase_transition.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            file_path = os.path.join(root, file)
            zipf.write(file_path, arcname=file)

print(f"\n追加検証が完了しました。結果を圧縮しました: {zip_filename}")
files.download(zip_filename)

<>:139: SyntaxWarning: invalid escape sequence '\k'
<>:143: SyntaxWarning: invalid escape sequence '\l'
<>:153: SyntaxWarning: invalid escape sequence '\k'
<>:155: SyntaxWarning: invalid escape sequence '\k'
<>:139: SyntaxWarning: invalid escape sequence '\k'
<>:143: SyntaxWarning: invalid escape sequence '\l'
<>:153: SyntaxWarning: invalid escape sequence '\k'
<>:155: SyntaxWarning: invalid escape sequence '\k'
/tmp/ipykernel_2834/1945168208.py:139: SyntaxWarning: invalid escape sequence '\k'
  label=f'D={D} ($\kappa$={df_fit[df_fit["Dimension_D"]==D]["Acceleration_Kappa"].values[0]:.2f})')
/tmp/ipykernel_2834/1945168208.py:143: SyntaxWarning: invalid escape sequence '\l'
  plt.xlabel("Sample Size $\log_{10}(N)$")
/tmp/ipykernel_2834/1945168208.py:153: SyntaxWarning: invalid escape sequence '\k'
  plt.title("Increase in Collapse Acceleration ($\kappa$) by Dimension")
/tmp/ipykernel_2834/1945168208.py:155: SyntaxWarning: invalid escape sequence '\k'
  plt.ylabel("Acceleration Parameter

崩壊曲線の相転移・加速度分析シミュレーションを開始します...

--- 次元数 D = 10 の相転移スキャン ---

--- 次元数 D = 50 の相転移スキャン ---

--- 次元数 D = 100 の相転移スキャン ---

--- 次元数 D = 200 の相転移スキャン ---


,Dimension_D,Acceleration_Kappa,Half_Life_Log10_N50,Transition_Width_DeltaLogN
0,10,18.029702,4.390997,0.326621
1,50,39.005734,5.105080,0.150975
2,100,50.000000,5.407729,0.117778
3,200,50.000000,5.702892,0.117778



追加検証が完了しました。結果を圧縮しました: hypothesis3_phase_transition.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>